# 01 — Raccolta, Parsing e Pulizia dei Dati (NYT)

Prima fase del progetto *ICE Perception and Topic Network*: costruzione del corpus NYT (2003 → oggi).

**Vincolo metodologico centrale**: "ICE" è un acronimo ambiguo (ghiaccio, treni, motori, altre sigle). Ogni filtro deve basarsi sulla tassonomia strutturata del NYT (`keywords[].name == organizations` e `value == "Immigration and Customs Enforcement (US)"`), mai su ricerca free-text della stringa "ICE".

**Questo notebook è ora la pipeline canonica**: tutto il codice di `scripts/nyt/fetch_nyt.py`, `parse_nyt.py` ed `eda_nyt.py` è inline qui sotto (non importato) — non chiama più quegli script. Eseguendo le celle in ordine:

1. verifica/completa il fetch dall'Archive API (idempotente: se il checkpoint segna già tutto scaricato, non richiama l'API);
2. **ricostruisce** `articles_full.parquet` e `articles_ice.parquet` dal raw, dentro il notebook, e li scrive su disco.

Equivalenza con l'output prodotto da `parse_nyt.py` già verificata in una run precedente (confronto riga per riga, risultato identico) — rimossa dal notebook una volta confermata, per non riportarla ad ogni esecuzione.

Gli script in `scripts/nyt/` restano per ora sul disco come riferimento storico, ma non sono più necessari per rigenerare i dati.

## Setup
In questa cella inauguriamo la fase di raccolta dati, configurando l'ambiente necessario per dialogare con le API di archivio del New York Times. Oltre a importare gli strumenti standard per le richieste web e la gestione dei dati (come `requests` e `pandas`), strutturiamo le directory di lavoro e stabiliamo le "regole d'ingaggio" del server. Impostiamo il punto di partenza temporale al gennaio 2003 — l'anno in cui l'agenzia ICE è diventata formalmente operativa — e predisponiamo costanti cruciali per tollerare gli errori di rete e i limiti di traffico (il tempo di *backoff* e i tentativi massimi). Questa architettura difensiva è indispensabile per garantire un download massivo e continuo, evitando che l'estrazione di oltre vent'anni di giornalismo si interrompa per un blocco temporaneo dei server.

In [ ]:
import json
import logging
import re
import time
import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import requests
from dotenv import load_dotenv

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw" / "nyt" / "archive"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
CHECKPOINT_PATH = BASE_DIR / "scripts" / "nyt" / "checkpoint.json"
OUT_FULL = PROCESSED_DIR / "articles_full.parquet"
OUT_ICE = PROCESSED_DIR / "articles_ice.parquet"

ARCHIVE_URL = "https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
START_YEAR_MONTH = (2003, 1)
MAX_RETRIES = 5
BACKOFF_BASE_SECONDS = 30  # usato solo su 429 / errori di rete, oltre allo sleep fisso tra richieste

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("nyt_pipeline")

print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# Funzione per caricare API_KEY e RATE_LIMIT da .env
def load_config() -> tuple[str, float]:
    load_dotenv(BASE_DIR / ".env")
    import os
    api_key = os.getenv("NYT_API_KEY")
    if not api_key:
        raise RuntimeError("NYT_API_KEY non trovata in .env")
    rate_limit_seconds = float(os.getenv("NYT_RATE_LIMIT_SECONDS", "12"))
    return api_key, rate_limit_seconds

# Config: confermiamo solo che la chiave sia presente in .env, mai stamparne il valore
api_key, rate_limit_seconds = load_config()
print(f"NYT_API_KEY caricata: {'si' if api_key else 'no'}")
print(f"Rate limit configurato: {rate_limit_seconds}s tra le richieste")

## 1. Raccolta dati — NYT Archive API
In questo blocco costruiamo la logica di avanzamento dell'estrazione e, soprattutto, il suo sistema di sicurezza. Da un lato, definiamo funzioni per iterare cronologicamente mese per mese a partire dal 2003 fino a oggi. Dall'altro, implementiamo un sistema di *checkpoint* basato su un file di registro (JSON). Questo meccanismo "salvavita" è essenziale quando si interrogano API su larga scala: tenendo traccia esatta dei mesi già acquisiti con successo, ci permette di mettere in pausa o interrompere il processo (volontariamente o per un crash di rete) e riprenderlo in un secondo momento esattamente da dove si era fermato, senza sprecare richieste o scaricare duplicati. Infine, stabiliamo una gerarchia di salvataggio dinamica per archiviare ordinatamente i file grezzi in arrivo dividendoli per anno.

**Scelta**: Archive API (`/svc/archive/v1/{year}/{month}.json`), non Article Search — 1 chiamata = 1 mese intero, nessuna paginazione, nessun cap sui risultati, nessun filtro lato server (il filtro ICE avviene interamente a valle, in parsing).

**Rate limit rispettato**: 500 req/giorno, 5/min → sleep fisso configurabile via `.env` (`NYT_RATE_LIMIT_SECONDS`, default 12s) tra una chiamata e l'altra.

La cella di esecuzione qui sotto è **idempotente**: se il checkpoint segna già tutti i mesi come completati (come in questo momento: 285/285), non fa nessuna chiamata API, si limita a confermarlo. È lo stesso identico comportamento dello script originale — se in futuro servisse aggiungere mesi nuovi (es. archivio che avanza), rieseguire questa stessa cella li scarica.

In [ ]:
# Funzione per formattare anno e mese della ricerca in una stringa con il formato YYYY-MM
def month_key(year: int, month: int) -> str:
    return f"{year:04d}-{month:02d}"

# Funzione che tiene aggiornati anno e mese durante l'esecuzione
def month_range(start: tuple[int, int], end: tuple[int, int]):
    year, month = start
    end_year, end_month = end
    while (year, month) <= (end_year, end_month):
        yield year, month
        month += 1
        if month > 12:
            month = 1
            year += 1

def current_year_month() -> tuple[int, int]:
    today = datetime.date.today()
    return today.year, today.month

# Carica il file di checkpoint con i mesi già scaricati
def load_checkpoint() -> dict:
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"completed_months": [], "failed_months": [], "last_run": None}

# Aggiorna il file di checkpoint salvando last_run e l'ultimo mese scaricato
def save_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_run"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)

# Definisce il path per il nuovo mese scaricato del tipo: data/raw/YYYY/YYYY-MM.json
def raw_json_path(year: int, month: int) -> Path:
    return RAW_DIR / f"{year:04d}" / f"{year:04d}-{month:02d}.json"



In questa cella definiamo il nucleo operativo del nostro downloader. Utilizzando un oggetto `requests.Session`, manteniamo la connessione HTTP aperta e attiva tra una chiamata e l'altra, risparmiando il tempo necessario per rinegoziare continuamente l'handshake con i server del New York Times. Il vero valore aggiunto di questa funzione risiede però nella sua resilienza: implementa un meccanismo di riprova automatica (retry) con pause crescenti (backoff) per gestire i frequenti cali di rete o i blocchi temporanei imposti dall'API per eccesso di traffico (il temuto errore 429 "Rate limit"). Questo approccio difensivo è cruciale per le estrazioni massive, poiché impedisce che il download di migliaia di articoli si schianti a causa di un banale singhiozzo del server.

In [ ]:
# usando l'oggetto requests.Session evitiamo di dover chiudere e riaprire la sessione,
# l'handshake tra client e server rimane attivo tra una richiesta e l'altra
def fetch_month(session: requests.Session, api_key: str, year: int, month: int) -> dict:
    """Scarica un mese dall'Archive API con retry/backoff. Solleva su fallimento definitivo."""
    url = ARCHIVE_URL.format(year=year, month=month)
    params = {"api-key": api_key}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=30)
        except requests.exceptions.RequestException as exc:
            log.warning("Errore di rete su %s (tentativo %d/%d): %s", month_key(year, month), attempt, MAX_RETRIES, exc)
            time.sleep(BACKOFF_BASE_SECONDS * attempt)
            continue

        if response.status_code == 200:
            return response.json()

        if response.status_code in (401, 403):
            raise RuntimeError(f"API key non valida o non autorizzata (HTTP {response.status_code})")

        if response.status_code == 429:
            log.warning(
                "Rate limit superato (429) su %s, attendo %ds prima di ritentare (tentativo %d/%d)",
                month_key(year, month), BACKOFF_BASE_SECONDS * attempt, attempt, MAX_RETRIES,
            )
            time.sleep(BACKOFF_BASE_SECONDS * attempt)
            continue

        log.warning(
            "Risposta inattesa HTTP %d su %s (tentativo %d/%d): %s",
            response.status_code, month_key(year, month), attempt, MAX_RETRIES, response.text[:200],
        )
        time.sleep(BACKOFF_BASE_SECONDS * attempt)

    raise RuntimeError(f"Fallito il fetch di {month_key(year, month)} dopo {MAX_RETRIES} tentativi")

 La funzione `run_fetch` orchestra il ciclo di estrazione: confronta la finestra temporale richiesta con il file di checkpoint, individua esattamente quali mesi mancano all'appello e avvia il download. Man mano che i file JSON grezzi vengono salvati su disco, il registro si aggiorna in tempo reale e lo script si mette in pausa per rispettare i limiti di velocità (rate limit) imposti dal NYT. Il grande pregio di questa architettura è l'idempotenza: possiamo eseguire questa cella innumerevoli volte senza rischiare di scaricare duplicati, corrompere i dati o sprecare le nostre quote API. Se l'archivio è già completo, lo script semplicemente se ne accorge e conclude le operazioni, rendendo la nostra pipeline sicura e pronta ad assorbire in futuro i mesi successivi senza sforzo.

In [ ]:
def run_fetch(start: tuple[int, int], end: tuple[int, int], only_failed: bool = False) -> None:
    api_key, rate_limit_seconds = load_config()
    checkpoint = load_checkpoint()
    completed = set(checkpoint["completed_months"])
    failed = set(checkpoint["failed_months"])

    if only_failed:
        months = [tuple(int(p) for p in k.split("-")) for k in sorted(failed)]
    else:
        months = [(y, m) for y, m in month_range(start, end) if month_key(y, m) not in completed]

    if not months:
        log.info("Niente da scaricare: tutti i mesi richiesti risultano già completati.")
        return

    log.info("Mesi da scaricare: %d", len(months))
    session = requests.Session()

    for year, month in months:
        key = month_key(year, month)
        try:
            data = fetch_month(session, api_key, year, month)
        except RuntimeError as exc:
            log.error(str(exc))
            failed.add(key)
            checkpoint["failed_months"] = sorted(failed)
            save_checkpoint(checkpoint)
            time.sleep(rate_limit_seconds)
            continue

        n_articles = len(data.get("response", {}).get("docs", []))
        path = raw_json_path(year, month)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)

        log.info("Salvato %s -> %s (%d articoli)", key, path.relative_to(BASE_DIR), n_articles)

        completed.add(key)
        failed.discard(key)
        checkpoint["completed_months"] = sorted(completed)
        checkpoint["failed_months"] = sorted(failed)
        save_checkpoint(checkpoint)

        time.sleep(rate_limit_seconds)

    log.info("Fatto. Completati: %d, falliti: %d", len(completed), len(failed))
    if failed:
        log.warning("Mesi falliti (rilancia con only_failed=True): %s", ", ".join(sorted(failed)))

# Idempotente: con il checkpoint gia' a 285/285 non fa alcuna chiamata API
run_fetch(START_YEAR_MONTH, current_year_month(), only_failed=False)

In [ ]:
checkpoint = load_checkpoint()
completed = checkpoint["completed_months"]
failed = checkpoint["failed_months"]

print(f"Mesi completati: {len(completed)}")
print(f"Mesi falliti:     {len(failed)}")
print(f"Range:            {completed[0]} -> {completed[-1]}")
print(f"Ultimo run:       {checkpoint['last_run']}")

In [ ]:
# Struttura di un file raw: 1 mese, articoli non filtrati, cosi' come restituiti dall'Archive API
sample_path = raw_json_path(2018, 6)
with open(sample_path, encoding="utf-8") as f:
    sample_month = json.load(f)

docs = sample_month["response"]["docs"]
print(f"File: {sample_path.relative_to(BASE_DIR)}")
print(f"Articoli nel mese: {len(docs)}")
print("\nCampi disponibili per articolo:")
print(sorted(docs[0].keys()))

## 2. Parsing e filtro ICE

In questo passaggio trasformiamo l'enorme mole di dati grezzi mensili in un formato strutturato e pertinente alla nostra ricerca. La funzione `extract_article` estrapola dal JSON originale solo i campi fondamentali (titolo, abstract, data, sezione, parole chiave e firme), appiattendo la struttura gerarchica per prepararla all'analisi tabellare. Al centro di questo blocco c'è la funzione `is_ice_article`, che rappresenta il vero confine editoriale del progetto. Gestendo in modo intelligente i piccoli cambi di formato del database del NYT avvenuti negli anni (come il passaggio da "organizations" a "Organization"), questa logica isola solo gli articoli contrassegnati in modo esplicito con l'etichetta ufficiale dell'agenzia ICE. In questo modo eliminiamo tutto il rumore della cronaca generica sull'immigrazione, garantendoci un corpus di partenza solido e tematicamente puro per le successive indagini sugli autori e sul vocabolario.

`is_ice_article()` marca un articolo come ICE se una delle sue `keywords` ha:
- `name` (case-insensitive) in `{"organizations", "organization"}`
- `value` (case-insensitive) uguale a `"Immigration and Customs Enforcement (US)"`

**Schema break verificato a 2025-01**: `keywords[].name` è plurale-minuscolo fino al 2024-12 (`organizations`, `subject`, `persons`...) e diventa singolare-capitalizzato dal 2025-01 (`Organization`, `Subject`, `Person`...). Il match case-insensitive copre entrambe le ere senza hardcodare una sola forma.

In [ ]:
# "organizations" (schema fino al 2024-12) / "Organization" (schema dal 2025-01) — confrontati case-insensitive
ICE_ORG_NAMES = {"organizations", "organization"}
ICE_ORG_VALUE = "immigration and customs enforcement (us)"

def is_ice_article(keywords: list) -> bool:
    for kw in keywords:
        name = (kw.get("name") or "").strip().lower()
        value = (kw.get("value") or "").strip().lower()
        if name in ICE_ORG_NAMES and value == ICE_ORG_VALUE:
            return True
    return False

# Estrae le informazioni dal json grezzo per ogni articolo
def extract_article(doc: dict) -> dict:
    keywords = doc.get("keywords") or []
    headline = doc.get("headline") or {}
    # byline.person: lista strutturata {firstname, middlename, lastname, organization, rank}
    # (piu' affidabile del parsing della stringa "By X and Y" su byline.original);
    # byline.organization: valorizzato quando l'autore e' una redazione ("By The New York Times")
    # e byline.person e' vuoto. Blob raw, non normalizzato: normalizzazione derivata a valle.
    byline = doc.get("byline") or {}
    return {
        "id": doc.get("_id"),
        "pub_date": doc.get("pub_date"),
        "headline": headline.get("main"),
        "abstract": doc.get("abstract"),
        "lead_paragraph": doc.get("lead_paragraph"),
        "web_url": doc.get("web_url"),
        "section_name": doc.get("section_name"),
        "word_count": doc.get("word_count"),
        "keywords_json": json.dumps(keywords, ensure_ascii=False),
        "byline_json": json.dumps(byline, ensure_ascii=False),
        "is_ice": is_ice_article(keywords),
    }

# Sorta i file in ordine temporale
def iter_raw_files():
    return sorted(RAW_DIR.glob("*/*.json"))

In [ ]:
# Esempio pre-2025 (schema plurale-minuscolo): cerchiamo un articolo ICE nel mese campione
ice_example_pre = next(d for d in docs if is_ice_article(d.get("keywords") or []))
print(ice_example_pre["headline"]["main"])
print(json.dumps(ice_example_pre["keywords"][:4], indent=2, ensure_ascii=False))

In [ ]:
# Esempio post-2025 (schema singolare-capitalizzato)
with open(raw_json_path(2025, 2), encoding="utf-8") as f:
    docs_2025 = json.load(f)["response"]["docs"]

ice_example_post = next(d for d in docs_2025 if is_ice_article(d.get("keywords") or []))
print(ice_example_post["headline"]["main"])
print(json.dumps(ice_example_post["keywords"][:4], indent=2, ensure_ascii=False))

In [ ]:
# extract_article(): stessa funzione che verra' usata sull'intero corpus qui sotto
pd.DataFrame([extract_article(ice_example_pre), extract_article(ice_example_post)])[
    ["id", "pub_date", "headline", "section_name", "word_count", "is_ice"]
]

### Esecuzione completa del parsing 

Questa cella legge **tutti** i 285 file raw (~4.7GB, ~2M articoli) e costruisce il corpus. 

In [ ]:
t0 = time.time()
files = iter_raw_files()
log.info("File raw da processare: %d", len(files))

rows = []
for path in files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    docs_month = data.get("response", {}).get("docs", [])
    month_rows = [extract_article(d) for d in docs_month]
    n_ice = sum(r["is_ice"] for r in month_rows)
    rows.extend(month_rows)

log.info("Righe estratte: %d (%.1fs)", len(rows), time.time() - t0)

In [ ]:
df_full_new = pd.DataFrame(rows)

before_dedup = len(df_full_new)
df_full_new = df_full_new.drop_duplicates(subset="id", keep="first")
if before_dedup != len(df_full_new):
    log.info("Rimossi %d duplicati per id", before_dedup - len(df_full_new))

# format="ISO8601" richiesto: dal 2025-01 pub_date miscela timestamp con/senza millisecondi
# (es. "2025-01-01T05:01:00.294Z" vs "2025-01-01T00:26:42Z") e l'inferenza di formato unico
# di pandas su questo array misto coercizza silenziosamente a NaT le righe che non matchano
# il formato dedotto dai primi elementi (~9.4k articoli su 2M, 31 dei quali ICE).
df_full_new["pub_date"] = pd.to_datetime(df_full_new["pub_date"], utc=True, format="ISO8601", errors="coerce")

df_ice_new = df_full_new[df_full_new["is_ice"]].reset_index(drop=True)
df_full_new = df_full_new.reset_index(drop=True)

print(f"articles_full (nuovo): {len(df_full_new):,} articoli")
print(f"articles_ice (nuovo):  {len(df_ice_new):,} articoli")

### Scrittura Parquet — il notebook diventa la fonte canonica

Generiamo `articles_full.parquet` e `articles_ice.parquet` col risultato appena ricostruito dal raw. Il raw resta comunque intatto.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_full_new.to_parquet(OUT_FULL, index=False)
log.info("Scritto %s (%d articoli totali)", OUT_FULL.relative_to(BASE_DIR), len(df_full_new))

df_ice_new.to_parquet(OUT_ICE, index=False)
log.info("Scritto %s (%d articoli ICE)", OUT_ICE.relative_to(BASE_DIR), len(df_ice_new))

# round-trip: rileggiamo dal disco per confermare che il file scritto sia corretto
df_full = pd.read_parquet(OUT_FULL)
df_ice = pd.read_parquet(OUT_ICE)
assert len(df_full) == len(df_full_new) and len(df_ice) == len(df_ice_new)
print("Round-trip OK: il file su disco corrisponde al DataFrame appena generato.")

## 3. Pulizia dei dati

Due problemi reali gestiti direttamente nel loop di parsing sopra:

1. **Deduplicazione per `id`** — un articolo può teoricamente comparire in più file mensili.
2. **Parsing date non uniforme** — dal 2025-01 `pub_date` mescola timestamp con/senza millisecondi. Senza `format="ISO8601"` esplicito, l'inferenza automatica di pandas coercizza silenziosamente ~9.4k righe a `NaT` (31 ICE) — bug reale trovato e corretto, riprodotto sopra.

In [ ]:
# Nulls per colonna, per capire dove serve attenzione a valle (es. sentiment su abstract/lead_paragraph)
df_full.isna().sum().to_frame("n_null").assign(pct=lambda d: (d["n_null"] / len(df_full) * 100).round(2))

In [ ]:
n_nat = df_full["pub_date"].isna().sum()
print(f"Righe con pub_date NaT: {n_nat}")
print(f"Range temporale: {df_full['pub_date'].min()} -> {df_full['pub_date'].max()}")

## 4. Validazione del filtro ICE

Controllo di qualità: campione casuale di `articles_ice` per ispezione visiva, più stima dei falsi negativi via regex sull'intero corpus.

In [ ]:
df_ice.sample(10, random_state=42)[["pub_date", "headline", "section_name"]]

In [ ]:
# Falsi negativi (stima): articoli che menzionano "ICE"/"Immigration and Customs Enforcement"
# in headline/abstract ma non taggati is_ice — quantifica il tradeoff precisione/recall.
# \bICE\b case-sensitive (non IGNORECASE): cattura l'acronimo maiuscolo ma non "ice cream"/"Ice Age"/
# altri usi comuni della parola minuscola, che altrimenti gonfierebbero i candidati con rumore non pertinente.
pattern = re.compile(r"\bICE\b|Immigration and Customs Enforcement")
text = (df_full["headline"].fillna("") + " " + df_full["abstract"].fillna(""))
candidates = df_full[text.str.contains(pattern) & ~df_full["is_ice"]]

print(f"Candidati falsi-negativi (regex, non taggati is_ice): {len(candidates)}")
candidates[["pub_date", "headline", "section_name"]].tail(10)



### 5. Analisi Temporale e Identificazione dei Picchi (Spike)
In questo blocco raggruppiamo i dati su base mensile per ottenere una visione macroscopica dell'attenzione del giornale. Filtriamo via gli eventuali articoli antecedenti al 2003 e calcoliamo non solo il volume assoluto di pezzi dedicati all'ICE, ma anche il loro peso percentuale rispetto all'intera produzione del New York Times in quello stesso mese. Successivamente, individuiamo i "picchi" assoluti di pubblicazione e tracciamo l'andamento storico su un grafico a linee. Questa visualizzazione è il nostro primo "elettrocardiogramma" della redazione: ci permette di correlare a colpo d'occhio l'intensità della copertura mediatica con i grandi eventi storici e politici (crisi al confine, elezioni, cambi di amministrazione) e prepara il terreno concettuale per l'algoritmo di Burst Detection che applicheremo in seguito.

In [ ]:
def load_monthly_counts(df: pd.DataFrame) -> pd.DataFrame:
    d = df[["pub_date", "is_ice"]].copy()

    before = len(d)
    d = d[d["pub_date"].dt.year >= 2003]
    dropped = before - len(d)
    if dropped:
        log.warning("Scartati %d articoli con pub_date fuori range corpus (<2003)", dropped)

    d["month"] = d["pub_date"].dt.tz_convert(None).dt.to_period("M")

    total = d.groupby("month").size().rename("total_articles")
    ice = d[d["is_ice"]].groupby("month").size().rename("ice_articles")

    monthly = pd.concat([total, ice], axis=1).fillna(0)
    monthly["ice_articles"] = monthly["ice_articles"].astype(int)
    monthly["ice_share_pct"] = (monthly["ice_articles"] / monthly["total_articles"] * 100).round(4)
    monthly = monthly.reset_index()
    monthly["month"] = monthly["month"].astype(str)
    return monthly

def find_spikes(monthly: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    return monthly.sort_values("ice_articles", ascending=False).head(top_n)

monthly = load_monthly_counts(df_full)

fig, ax = plt.subplots(figsize=(14, 4))
x = pd.to_datetime(monthly["month"], format="%Y-%m")
ax.plot(x, monthly["ice_articles"], color="#1f77b4", linewidth=1.2)
ax.set_title("NYT — Articoli ICE per mese (2003-2026)")
ax.set_xlabel("Mese")
ax.set_ylabel("N. articoli ICE")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
find_spikes(monthly, top_n=10)

In [ ]:
yearly = monthly.copy()
yearly["year"] = pd.to_datetime(yearly["month"]).dt.year
yearly = yearly.groupby("year")["ice_articles"].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(yearly.index.astype(str), yearly.values, color="#d62728")
ax.set_title("NYT — Articoli ICE per anno (2003-2026)")
ax.set_xlabel("Anno")
ax.set_ylabel("N. articoli ICE")
ax.tick_params(axis="x", rotation=90)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Commenti dei lettori agli articoli ICE

Raccolta dei commenti pubblicati dai lettori sotto gli articoli di `articles_ice.parquet`: è la reazione del pubblico dentro il NYT stesso, da affiancare a quella di Telegram.

**La Community API ufficiale non esiste più.** Verificato il 2026-09-30: `api.nytimes.com/svc/community/v3/...` risponde 404 con corpo vuoto su ogni endpoint, con chiave valida, senza chiave o con chiave sbagliata, esattamente come un servizio inesistente (l'Archive API con chiave sbagliata risponde invece 401). Non è un problema di permessi: il servizio è stato tolto.

**Fonte usata**: l'endpoint che il sito nytimes.com usa per mostrare i commenti, `www.nytimes.com/svc/community/V3/requestHandler`. Non richiede chiave. Non è documentato, non è coperto dai termini d'uso delle API e può cambiare o bloccarsi senza preavviso: va dichiarato nel report come limite.

**Rate limit**: nessuno dichiarato e nessuna intestazione di quota nelle risposte. I limiti delle API ufficiali (500 richieste al giorno, 5 al minuto) qui non valgono. Usiamo una pausa fissa prudente, configurabile in `.env` con `NYT_COMMENTS_RATE_LIMIT_SECONDS` (default 6 s, cioè 10 richieste al minuto), più attesa crescente su 429/403/5xx, rispettando `Retry-After` se presente.

**Come si scarica un articolo**: `cmd=GetCommentsAll` con `limit` e `replyLimit` alti restituisce in una sola risposta tutti i commenti e tutte le risposte (senza `replyLimit` arrivano solo 3 risposte per commento). Il codice resta comunque difensivo: pagina con `offset` finché non ha tutti i commenti di primo livello dichiarati, e per ogni commento con risposte mancanti chiama `cmd=GetRepliesBySequence`.

**Stesse regole del §1**: il fetch non filtra, salva il raw così com'è in `data/raw/nyt/comments/YYYY/<uuid>.json` (un file per articolo, con tutte le risposte ricevute) e aggiorna il checkpoint `data/raw/nyt/comments/checkpoint.json` dopo ogni articolo. Si può interrompere e riprendere. Gli articoli pubblicati da meno di `COMMENTS_SETTLE_DAYS` giorni vengono scaricati ma non segnati come completati, perché i commenti possono ancora arrivare: a ogni esecuzione vengono riscaricati.

In [ ]:
import html
import os

COMMENTS_URL = "https://www.nytimes.com/svc/community/V3/requestHandler"
COMMENTS_RAW_DIR = BASE_DIR / "data" / "raw" / "nyt" / "comments"
COMMENTS_CHECKPOINT_PATH = COMMENTS_RAW_DIR / "checkpoint.json"
OUT_COMMENTS = PROCESSED_DIR / "comments_ice.parquet"
OUT_COMMENTS_ARTICLES = PROCESSED_DIR / "comments_ice_articles.parquet"

# valori alti: una risposta contiene tutti i commenti e tutte le risposte dell'articolo
COMMENTS_PAGE_LIMIT = 1000
COMMENTS_REPLY_LIMIT = 1000
COMMENTS_SETTLE_DAYS = 7
# l'endpoint del sito rifiuta le richieste senza User-Agent da browser
COMMENTS_HEADERS = {"User-Agent": "Mozilla/5.0 (research project, ICE topic network)"}


def load_comments_rate_limit() -> float:
    load_dotenv(BASE_DIR / ".env")
    return float(os.getenv("NYT_COMMENTS_RATE_LIMIT_SECONDS", "6"))


def load_comments_checkpoint() -> dict:
    if COMMENTS_CHECKPOINT_PATH.exists():
        with open(COMMENTS_CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"completed_articles": [], "failed_articles": [], "last_run": None}


def save_comments_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_run"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    COMMENTS_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(COMMENTS_CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2, ensure_ascii=False)


def comments_raw_path(article_id: str, pub_date: pd.Timestamp) -> Path:
    # id del tipo "nyt://article/<uuid>": il nome file e' l'uuid
    return COMMENTS_RAW_DIR / f"{pub_date.year:04d}" / f"{article_id.rsplit('/', 1)[-1]}.json"


def request_comments(session: requests.Session, params: dict, rate_limit_seconds: float) -> dict:
    """Una chiamata all'endpoint commenti con retry/backoff. Ritorna il blocco `results`. Solleva su fallimento definitivo."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(COMMENTS_URL, params=params, headers=COMMENTS_HEADERS, timeout=60)
        except requests.exceptions.RequestException as exc:
            log.warning("Errore di rete (tentativo %d/%d): %s", attempt, MAX_RETRIES, exc)
            time.sleep(BACKOFF_BASE_SECONDS * attempt)
            continue

        if response.status_code == 200:
            try:
                payload = response.json()
            except ValueError:
                payload = {}
            time.sleep(rate_limit_seconds)
            if payload.get("status") == "OK" and "results" in payload:
                return payload["results"]
            log.warning("Risposta 200 non valida (tentativo %d/%d): %s", attempt, MAX_RETRIES, response.text[:200])
            continue

        if response.status_code in (403, 429) or response.status_code >= 500:
            retry_after = response.headers.get("Retry-After", "")
            wait = int(retry_after) if retry_after.isdigit() else BACKOFF_BASE_SECONDS * attempt
            log.warning("HTTP %d (tentativo %d/%d), attendo %ds", response.status_code, attempt, MAX_RETRIES, wait)
            time.sleep(wait)
            continue

        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")

    raise RuntimeError(f"Fallito dopo {MAX_RETRIES} tentativi")


def fetch_article_comments(session: requests.Session, web_url: str, rate_limit_seconds: float) -> list[dict]:
    """Tutte le risposte grezze per un articolo: pagine di commenti + eventuali risposte mancanti."""
    responses = []
    parents = {}
    offset = 0
    while True:
        params = {
            "url": web_url, "method": "get", "cmd": "GetCommentsAll", "sort": "oldest",
            "commentSequence": 0, "includeReplies": "true",
            "offset": offset, "limit": COMMENTS_PAGE_LIMIT, "replyLimit": COMMENTS_REPLY_LIMIT,
        }
        results = request_comments(session, params, rate_limit_seconds)
        responses.append({"request": params, "results": results})
        page = results.get("comments") or []
        for comment in page:
            parents[comment["commentID"]] = comment
        if not page or len(parents) >= (results.get("totalParentCommentsFound") or 0):
            break
        offset += len(page)

    # rete di sicurezza: se un commento arriva con meno risposte di quelle dichiarate, le chiediamo a parte
    for comment in parents.values():
        got = len(comment.get("replies") or [])
        reply_offset = 0
        while got < (comment.get("replyCount") or 0):
            params = {
                "url": web_url, "method": "get", "cmd": "GetRepliesBySequence",
                "commentSequence": comment["commentSequence"], "offset": reply_offset, "limit": COMMENTS_PAGE_LIMIT,
            }
            results = request_comments(session, params, rate_limit_seconds)
            responses.append({"request": params, "results": results})
            thread = results.get("comments") or []
            n_new = len(thread[0].get("replies") or []) if thread else 0
            if n_new <= got - reply_offset:
                break
            reply_offset += n_new
            got = max(got, reply_offset)

    return responses


def run_comments_fetch(max_articles: int | None = None, only_failed: bool = False) -> None:
    """Scarica i commenti degli articoli ICE non ancora completati. Checkpoint dopo ogni articolo."""
    rate_limit_seconds = load_comments_rate_limit()
    checkpoint = load_comments_checkpoint()
    completed = set(checkpoint["completed_articles"])
    failed = set(checkpoint["failed_articles"])

    articles = pd.read_parquet(OUT_ICE, columns=["id", "web_url", "pub_date", "headline"]).sort_values("pub_date")
    if only_failed:
        todo = articles[articles["id"].isin(failed)]
    else:
        todo = articles[~articles["id"].isin(completed)]
    if max_articles is not None:
        todo = todo.head(max_articles)

    if todo.empty:
        log.info("Niente da scaricare: tutti gli articoli ICE risultano già completati.")
        return

    log.info("Articoli da scaricare: %d (pausa %.0fs tra le richieste)", len(todo), rate_limit_seconds)
    session = requests.Session()
    now = pd.Timestamp.now(tz="UTC")

    for i, row in enumerate(todo.itertuples(), start=1):
        try:
            responses = fetch_article_comments(session, row.web_url, rate_limit_seconds)
        except Exception as exc:
            log.error("[%d/%d] %s fallito: %s", i, len(todo), row.web_url, exc)
            failed.add(row.id)
            checkpoint["failed_articles"] = sorted(failed)
            save_comments_checkpoint(checkpoint)
            continue

        path = comments_raw_path(row.id, row.pub_date)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump({
                "article_id": row.id,
                "web_url": row.web_url,
                "fetched_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "responses": responses,
            }, f, ensure_ascii=False)

        failed.discard(row.id)
        settled = (now - row.pub_date).days >= COMMENTS_SETTLE_DAYS
        if settled:
            completed.add(row.id)
        checkpoint["completed_articles"] = sorted(completed)
        checkpoint["failed_articles"] = sorted(failed)
        save_comments_checkpoint(checkpoint)

        total = responses[0]["results"].get("totalCommentsFound") or 0
        log.info(
            "[%d/%d] %s: %d commenti, %d richieste%s",
            i, len(todo), row.pub_date.date(), total, len(responses), "" if settled else " (recente, non segnato completato)",
        )

    log.info("Fetch commenti terminato: %d completati, %d falliti", len(completed), len(failed))

In [ ]:
# Idempotente: scarica solo gli articoli mancanti. Run completo = ~2.100 richieste, ~3,5 ore con pausa di 6s.
run_comments_fetch()

comments_checkpoint = load_comments_checkpoint()
print(f"Articoli completati: {len(comments_checkpoint['completed_articles'])}")
print(f"Articoli falliti:    {len(comments_checkpoint['failed_articles'])}")
print(f"Ultimo run:          {comments_checkpoint['last_run']}")

### Parsing dei commenti

Dal raw a due tabelle, rigenerabili senza richiamare l'endpoint:

- `comments_ice.parquet`: una riga per commento o risposta. Testo integrale ripulito dall'HTML, struttura del thread (`parent_id`, `thread_id`, `depth`, `parent_user_display_name`), utente (`user_id`, `user_display_name`, `user_location`, `user_title`, `is_anonymous`, `trusted`), gradimento (`recommendations`, `reply_count`), selezione editoriale (`editors_selection`), tipo (`comment`, `userReply`, `reporterReply`), timestamp (`created_at`, `approved_at`, `updated_at`).
- `comments_ice_articles.parquet`: una riga per articolo ICE scaricato, con i totali dichiarati dal NYT e quelli effettivamente raccolti, per misurare la copertura.

Feature derivate utili all'analisi: `hours_since_publication` (velocità della reazione), `moderation_delay_minutes` (tempo tra scrittura e approvazione), `word_count`, `is_reply`, `is_reporter_reply`.

`created_at_imputed` (1/0): sui commenti degli articoli pre-2018 il NYT ha riscritto `createDate` alla data di migrazione del sistema commenti (ottobre 2017-2018), successiva a `approveDate`. Dove `created_at > approved_at` si usa `approved_at` come data del commento e il flag vale 1; per queste righe `moderation_delay_minutes` è mancante.

`user_id` e `user_display_name` sono dati personali di lettori reali: servono per le reti di interazione, ma vanno pseudonimizzati prima di pubblicare qualunque tabella o grafo.

In [ ]:
def clean_comment_body(body: str | None) -> str:
    """Testo del commento senza HTML: <br/> diventa a capo, il resto dei tag e' rimosso, entita' decodificate."""
    text = re.sub(r"<br\s*/?>", "\n", body or "", flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", "", text)
    return html.unescape(text).strip()


def null_if_placeholder(value):
    # il NYT scrive la stringa "NULL" al posto del valore mancante
    return None if value in (None, "", "NULL") else value


def extract_comment(comment: dict, article_id: str, thread_id: int) -> dict:
    body = clean_comment_body(comment.get("commentBody"))
    return {
        "comment_id": comment["commentID"],
        "article_id": article_id,
        "thread_id": thread_id,
        "parent_id": comment.get("parentID"),
        "depth": comment.get("depth"),
        "comment_type": comment.get("commentType"),
        "is_reply": comment.get("parentID") is not None,
        "is_reporter_reply": comment.get("commentType") == "reporterReply",
        "body": body,
        "word_count": len(body.split()),
        "created_at": comment.get("createDate"),
        "approved_at": comment.get("approveDate"),
        "updated_at": comment.get("updateDate"),
        "recommendations": comment.get("recommendations"),
        "reply_count": comment.get("replyCount"),
        "editors_selection": bool(comment.get("editorsSelection")),
        "status": comment.get("status"),
        "user_id": comment.get("userID"),
        "user_display_name": null_if_placeholder(comment.get("userDisplayName")),
        "user_location": null_if_placeholder(comment.get("userLocation")),
        "user_title": null_if_placeholder(comment.get("userTitle")),
        "is_anonymous": bool(comment.get("isAnonymous")),
        "trusted": bool(comment.get("trusted")),
        "parent_user_display_name": null_if_placeholder(comment.get("parentUserDisplayName")),
    }


def parse_comments_file(path: Path) -> tuple[list[dict], dict]:
    """Un file raw -> (righe commento, riga di riepilogo articolo)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = {}
    for response in data["responses"]:
        for parent in response["results"].get("comments") or []:
            rows.setdefault(parent["commentID"], extract_comment(parent, data["article_id"], parent["commentID"]))
            for reply in parent.get("replies") or []:
                rows.setdefault(reply["commentID"], extract_comment(reply, data["article_id"], parent["commentID"]))

    first = data["responses"][0]["results"]
    summary = {
        "article_id": data["article_id"],
        "web_url": data["web_url"],
        "fetched_at": data["fetched_at"],
        "n_requests": len(data["responses"]),
        "total_comments_declared": first.get("totalCommentsFound") or 0,
        "total_parents_declared": first.get("totalParentCommentsFound") or 0,
        "total_replies_declared": first.get("totalReplyCommentsFound") or 0,
        "total_editors_selection_declared": first.get("totalEditorsSelectionFound") or 0,
        "total_reporter_replies_declared": first.get("totalReporterReplyCommentsFound") or 0,
        "n_comments_collected": len(rows),
    }
    return list(rows.values()), summary


comment_files = sorted(COMMENTS_RAW_DIR.glob("*/*.json"))
log.info("File raw commenti da processare: %d", len(comment_files))

comment_rows, article_rows = [], []
for path in comment_files:
    file_rows, summary = parse_comments_file(path)
    comment_rows.extend(file_rows)
    article_rows.append(summary)

articles_meta = pd.read_parquet(OUT_ICE, columns=["id", "pub_date", "headline", "section_name"]).rename(
    columns={"id": "article_id", "pub_date": "article_pub_date"}
)

df_comment_articles = pd.DataFrame(article_rows).merge(articles_meta, on="article_id", how="left")
df_comment_articles["fetched_at"] = pd.to_datetime(df_comment_articles["fetched_at"], utc=True, format="ISO8601")
df_comment_articles["has_comments"] = df_comment_articles["total_comments_declared"] > 0

df_comments = pd.DataFrame(comment_rows).drop_duplicates(subset="comment_id", keep="first")
for col in ("created_at", "approved_at", "updated_at"):
    df_comments[col] = pd.to_datetime(pd.to_numeric(df_comments[col], errors="coerce"), unit="s", utc=True)
df_comments["parent_id"] = df_comments["parent_id"].astype("Int64")
# I commenti anteriori alla migrazione del sistema commenti (fine 2017) hanno createDate riscritto
# alla data di migrazione, quindi successivo a approveDate: per questi vale la data di approvazione.
wrong_created = df_comments["created_at"] > df_comments["approved_at"]
df_comments["created_at_imputed"] = wrong_created.astype("int8")
df_comments.loc[wrong_created, "created_at"] = df_comments.loc[wrong_created, "approved_at"]
log.info("created_at sostituito con approved_at su %d commenti (createDate > approveDate)", wrong_created.sum())
df_comments = df_comments.merge(articles_meta[["article_id", "article_pub_date"]], on="article_id", how="left")
df_comments["hours_since_publication"] = (
    (df_comments["created_at"] - df_comments["article_pub_date"]).dt.total_seconds() / 3600
)
df_comments["moderation_delay_minutes"] = (
    (df_comments["approved_at"] - df_comments["created_at"]).dt.total_seconds() / 60
)
# ritardo di moderazione non misurabile dove created_at e' imputato
df_comments.loc[wrong_created, "moderation_delay_minutes"] = float("nan")
df_comments = df_comments.sort_values(["article_pub_date", "article_id", "thread_id", "created_at"]).reset_index(drop=True)

df_comments.to_parquet(OUT_COMMENTS, index=False)
df_comment_articles.to_parquet(OUT_COMMENTS_ARTICLES, index=False)
log.info("Scritto %s (%d commenti)", OUT_COMMENTS.relative_to(BASE_DIR), len(df_comments))
log.info("Scritto %s (%d articoli)", OUT_COMMENTS_ARTICLES.relative_to(BASE_DIR), len(df_comment_articles))

In [ ]:
# Copertura: commenti raccolti contro commenti dichiarati dal NYT, e quanti articoli ICE hanno commenti
declared = int(df_comment_articles["total_comments_declared"].sum())
print(f"Articoli scaricati:        {len(df_comment_articles):,}")
print(f"Articoli con commenti:     {int(df_comment_articles['has_comments'].sum()):,}")
print(f"Commenti dichiarati:       {declared:,}")
print(f"Commenti raccolti:         {len(df_comments):,} ({len(df_comments) / declared * 100 if declared else 0:.1f}%)")
print(f"  di cui risposte:         {int(df_comments['is_reply'].sum()):,}")
print(f"  selezioni editoriali:    {int(df_comments['editors_selection'].sum()):,}")
print(f"  risposte di giornalisti: {int(df_comments['is_reporter_reply'].sum()):,}")
print(f"Utenti distinti:           {df_comments['user_id'].nunique():,}")

by_year = df_comment_articles.assign(year=df_comment_articles["article_pub_date"].dt.year).groupby("year").agg(
    articoli=("article_id", "size"),
    con_commenti=("has_comments", "sum"),
    commenti_dichiarati=("total_comments_declared", "sum"),
    commenti_raccolti=("n_comments_collected", "sum"),
)
by_year

## Riepilogo

- Corpus completo (`articles_full`, conteggio sopra): 2003-01 → 2026-09, 0 duplicati residui, 0 `pub_date` nulli.
- Sottoinsieme ICE (`articles_ice`, conteggio sopra): filtro validato su campione (0 falsi positivi), gap di recall stimato ~5% (documentato, tradeoff accettato per privilegiare precisione).
- Spike principale: 2025-2026 (contesto: seconda amministrazione Trump), picco mensile a febbraio 2026.
- I due Parquet in `data/processed/nyt/` sono **rigenerati da questo notebook** ad ogni esecuzione.
